# GENERATOR VISUALISASI - PT Mensana Aneka Satwa
## Gaya Visual: Sama dengan Dashboard (Colorful, Elegan, Siap Presentasi)

---

Jalankan seluruh cell untuk menghasilkan gambar PNG di folder `charts/`.

In [1]:
# ============================================================
# IMPORT & SETUP
# ============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import timedelta
import os

os.makedirs('charts', exist_ok=True)

# ============================================================
# DESIGN SYSTEM (sama dengan dashboard.py)
# ============================================================
COLORS = {
    'primary': '#0F766E', 'primary_light': '#14B8A6',
    'primary_dark': '#0D4F4F',
    'accent_gold': '#F59E0B', 'accent_coral': '#F97316',
    'accent_blue': '#3B82F6', 'accent_purple': '#8B5CF6',
    'accent_rose': '#F43F5E', 'text': '#1E293B', 'text_muted': '#64748B',
    'grid': '#E2E8F0', 'bg': 'white',
}
CHART_COLORS = ['#14B8A6', '#F59E0B', '#3B82F6', '#F97316', '#8B5CF6',
                '#F43F5E', '#06B6D4', '#84CC16', '#EC4899', '#6366F1']

# Chart template
CHART_FONT = dict(family='Arial, Helvetica, sans-serif', size=12, color=COLORS['text'])
def styled(fig, height=450, width=900):
    fig.update_layout(
        font=CHART_FONT,
        paper_bgcolor=COLORS['bg'],
        plot_bgcolor=COLORS['bg'],
        margin=dict(l=50, r=30, t=55, b=40),
        height=height, width=width,
        title=dict(font=dict(size=15, color=COLORS['text'], family='Arial, Helvetica, sans-serif'), x=0, xanchor='left'),
        xaxis=dict(gridcolor=COLORS['grid'], zeroline=False),
        yaxis=dict(gridcolor=COLORS['grid'], zeroline=False),
    )
    return fig

def save(fig, name):
    fig.write_image(f'charts/{name}.png', scale=2)
    print(f'  Saved: {name}.png')

print('Design system loaded.')

Design system loaded.


In [2]:
# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_excel('Data_Mentah_Harian.xlsx', sheet_name='Data_Penjualan_Harian')
df['Tanggal'] = pd.to_datetime(df['Tanggal'])
month_order = ['Januari', 'Februari', 'Maret']
day_order = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
total_omset = df['Omset'].sum()

print(f'Total Records: {len(df):,}')
print(f'Total Omset: Rp {total_omset:,.0f}')
print(f'Cabang: {df["Nama_Cabang"].nunique()} | Produk: {df["Nama_Standar"].nunique()}')

Total Records: 13,500
Total Omset: Rp 2,320,386,997
Cabang: 10 | Produk: 15


---
## VISUALISASI

In [3]:
# 1. OMSET PER CABANG PER BULAN (Grouped Bar)
df_bulanan = df.groupby(['Bulan', 'Nama_Cabang'])['Omset'].sum().reset_index()
df_bulanan['Bulan'] = pd.Categorical(df_bulanan['Bulan'], categories=month_order, ordered=True)

fig1 = px.bar(df_bulanan, x='Nama_Cabang', y='Omset', color='Bulan', barmode='group',
              color_discrete_map={'Januari': COLORS['primary'], 'Februari': COLORS['accent_gold'], 'Maret': COLORS['accent_blue']},
              labels={'Omset': 'Total Omset (Rp)', 'Nama_Cabang': ''})
fig1.update_layout(legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center'))
styled(fig1, 480)
fig1.show()
save(fig1, '01_omset_cabang_bulanan')

  Saved: 01_omset_cabang_bulanan.png


In [4]:
# 2. DONUT DISTRIBUSI CABANG
branch_total = df.groupby('Nama_Cabang')['Omset'].sum().sort_values(ascending=False)

fig2 = px.pie(values=branch_total.values, names=branch_total.index, hole=0.5,
              color_discrete_sequence=CHART_COLORS)
fig2.update_traces(textposition='inside', textinfo='percent+label', textfont_size=11,
                   hovertemplate='<b>%{label}</b><br>Rp %{value:,.0f}<br>%{percent}<extra></extra>')
fig2.update_layout(showlegend=False,
                   annotations=[dict(text=f'<b>{len(branch_total)}</b><br>Cabang', x=0.5, y=0.5,
                                    font_size=14, showarrow=False, font_color=COLORS['text'])])
styled(fig2, 450)
fig2.show()
save(fig2, '02_donut_cabang')

  Saved: 02_donut_cabang.png


In [5]:
# 3. TOP 10 PRODUK TERLARIS
product_ranking = df.groupby(['Nama_Standar', 'Kategori']).agg(
    Total_Omset=('Omset', 'sum')).reset_index().sort_values('Total_Omset', ascending=False).head(10)

cat_map = dict(zip(product_ranking['Kategori'].unique(), CHART_COLORS))
fig3 = px.bar(product_ranking, x='Total_Omset', y='Nama_Standar', color='Kategori',
              orientation='h', color_discrete_map=cat_map,
              labels={'Total_Omset': 'Total Omset (Rp)', 'Nama_Standar': ''})
fig3.update_traces(texttemplate='Rp %{x:,.0f}', textposition='outside', textfont_size=10,
                   textfont_color=COLORS['text_muted'])
fig3.update_layout(yaxis=dict(categoryorder='total ascending'), margin=dict(r=120))
styled(fig3, 420)
fig3.show()
save(fig3, '03_top10_produk')

  Saved: 03_top10_produk.png


In [6]:
# 4. TOTAL OMSET PER KATEGORI
cat_bar = df.groupby('Kategori')['Omset'].sum().sort_values(ascending=True).reset_index()
cat_bar.columns = ['Kategori', 'Total_Omset']
cat_colors = [COLORS['accent_blue'], COLORS['accent_coral'], COLORS['accent_gold'],
              COLORS['accent_rose'], COLORS['accent_purple'], COLORS['primary']][:len(cat_bar)]

fig4 = px.bar(cat_bar, x='Total_Omset', y='Kategori', orientation='h',
              labels={'Total_Omset': 'Total Omset (Rp)', 'Kategori': ''})
fig4.update_traces(marker_color=cat_colors, texttemplate='Rp %{x:,.0f}',
                   textposition='outside', textfont_size=11, textfont_color=COLORS['text_muted'])
fig4.update_layout(margin=dict(r=120))
styled(fig4, 380)
fig4.show()
save(fig4, '04_kategori_bar')

  Saved: 04_kategori_bar.png


In [7]:
# 5. TREN OMSET HARIAN + GARIS REGRESI
daily = df.groupby('Tanggal')['Omset'].sum().reset_index()
daily['day_num'] = (daily['Tanggal'] - daily['Tanggal'].min()).dt.days
coeffs = np.polyfit(daily['day_num'], daily['Omset'], 1)
trend_fn = np.poly1d(coeffs)
daily['Trendline'] = trend_fn(daily['day_num'])

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=daily['Tanggal'], y=daily['Omset'], mode='lines',
                          name='Omset Harian', line=dict(color=COLORS['primary'], width=1.5),
                          fill='tozeroy', fillcolor='rgba(20,184,166,0.08)',
                          hovertemplate='<b>%{x|%d %b %Y}</b><br>Rp %{y:,.0f}<extra></extra>'))
fig5.add_trace(go.Scatter(x=daily['Tanggal'], y=daily['Trendline'], mode='lines',
                          name='Trendline Regresi',
                          line=dict(color=COLORS['accent_rose'], width=2.5, dash='dash')))
fig5.update_layout(xaxis_title='', yaxis_title='Omset (Rp)',
                   legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center'))
styled(fig5, 420)
fig5.show()
save(fig5, '05_tren_harian_regresi')

  Saved: 05_tren_harian_regresi.png


In [8]:
# 6. HARI KERJA vs AKHIR PEKAN
df['Tipe_Hari'] = df['Hari'].apply(lambda x: 'Akhir Pekan' if x in ['Sabtu', 'Minggu'] else 'Hari Kerja')
dow_compare = df.groupby('Tipe_Hari')['Omset'].mean().reset_index()
dow_compare.columns = ['Tipe_Hari', 'Rata_rata']

fig6 = px.bar(dow_compare, x='Tipe_Hari', y='Rata_rata', color='Tipe_Hari',
              color_discrete_map={'Hari Kerja': COLORS['primary'], 'Akhir Pekan': COLORS['accent_coral']},
              labels={'Rata_rata': 'Rata-rata Omset (Rp)', 'Tipe_Hari': ''})
fig6.update_traces(texttemplate='Rp %{y:,.0f}', textposition='outside', textfont_size=11)
fig6.update_layout(showlegend=False, yaxis=dict(zeroline=False))
styled(fig6, 400)
fig6.show()
save(fig6, '06_hari_kerja_weekend')

  Saved: 06_hari_kerja_weekend.png


In [9]:
# 7. OMSET PER HARI DALAM SEMINGGU
dow_daily = df.groupby('Hari')['Omset'].sum().reindex(day_order).reset_index()
dow_colors = [COLORS['primary']] * 5 + [COLORS['accent_coral']] * 2

fig7 = px.bar(dow_daily, x='Hari', y='Omset', labels={'Omset': 'Total Omset (Rp)', 'Hari': ''})
fig7.update_traces(marker_color=dow_colors, texttemplate='Rp %{y:,.0f}',
                   textposition='outside', textfont_size=9, textfont_color=COLORS['text_muted'])
fig7.update_layout(yaxis=dict(zeroline=False))
styled(fig7, 420)
fig7.show()
save(fig7, '07_omset_per_hari')

  Saved: 07_omset_per_hari.png


In [10]:
# 8. DAMPAK HARI LIBUR NASIONAL
holiday_names = {'2026-01-01': 'Tahun Baru', '2026-01-29': "Isra Mi'raj",
                 '2026-02-17': 'Tahun Baru Imlek', '2026-03-19': 'Nyepi'}

# Rata-rata omset hari kerja normal (tanpa libur & tanpa weekend)
normal_days = df[
    (~df['Tanggal'].dt.strftime('%Y-%m-%d').isin(holiday_names.keys())) &
    (~df['Hari'].isin(['Sabtu', 'Minggu']))
]
hol_labels = ['Hari Kerja Normal']
hol_values = [normal_days.groupby('Tanggal')['Omset'].sum().mean()]

for d, n in holiday_names.items():
    t = pd.Timestamp(d)
    if t in df['Tanggal'].values:
        hol_labels.append(n)
        hol_values.append(df[df['Tanggal'] == t]['Omset'].sum())

fig8 = px.bar(x=hol_labels, y=hol_values, labels={'x': '', 'y': 'Omset (Rp)'})
fig8.update_traces(marker_color=[COLORS['primary']] + [COLORS['accent_coral']] * len(holiday_names),
                   texttemplate='Rp %{y:,.0f}', textposition='outside', textfont_size=10)
fig8.update_layout(yaxis=dict(zeroline=False))
styled(fig8, 420)
fig8.show()
save(fig8, '08_dampak_libur')

  Saved: 08_dampak_libur.png


In [11]:
# 9. PROYEKSI 1 BULAN + GARIS REGRESI
monthly_omset = df.groupby('Bulan')['Omset'].sum()
last_date = df['Tanggal'].max()
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
forecast_day_nums = np.array([(d - daily['Tanggal'].min()).days for d in forecast_dates])
forecast_values = trend_fn(forecast_day_nums)
apr_proj = forecast_values.sum()

month_start_days = {'Januari': 0, 'Februari': 31, 'Maret': 59, 'April': 90}
month_n_days = {'Januari': 31, 'Februari': 28, 'Maret': 31, 'April': 30}
months = ['Januari', 'Februari', 'Maret', 'April\n(Proyeksi)']
bar_h = [monthly_omset['Januari'], monthly_omset['Februari'], monthly_omset['Maret'], apr_proj]
trend_s = [trend_fn(month_start_days[m]) * month_n_days[m] for m in month_start_days]

fig9 = go.Figure()
fig9.add_trace(go.Bar(x=months, y=bar_h, name='Omset',
                      marker_color=[COLORS['primary']]*3 + [COLORS['accent_gold']],
                      text=[f'Rp {v:,.0f}' for v in bar_h], textposition='outside', textfont_size=10))
fig9.add_trace(go.Scatter(x=months, y=trend_s, name='Garis Regresi',
                          mode='lines', line=dict(color=COLORS['accent_rose'], width=3)))
fig9.update_layout(xaxis_title='', yaxis_title='Omset (Rp)', yaxis=dict(rangemode='tozero'),
                   legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center'))
styled(fig9, 450)
fig9.show()
save(fig9, '09_proyeksi_april')

  Saved: 09_proyeksi_april.png


In [12]:
# 10. OMOSET KATEGORI PER BULAN (Stacked Bar)
cat_monthly = df.groupby(['Bulan', 'Kategori'])['Omset'].sum().reset_index()
cat_monthly['Bulan'] = pd.Categorical(cat_monthly['Bulan'], categories=month_order, ordered=True)
cat_monthly = cat_monthly.sort_values('Bulan')

fig10 = px.bar(cat_monthly, x='Bulan', y='Omset', color='Kategori',
               color_discrete_sequence=CHART_COLORS[:len(cat_monthly['Kategori'].unique())],
               labels={'Omset': 'Total Omset (Rp)', 'Bulan': ''})
fig10.update_layout(barmode='stack', legend=dict(orientation='h', y=1.15, x=0.5, xanchor='center'))
styled(fig10, 450)
fig10.show()
save(fig10, '10_kategori_stacked')

  Saved: 10_kategori_stacked.png


In [13]:
# 11. BOX PLOT DISTRIBUSI PER CABANG
daily_branch = df.groupby(['Tanggal', 'Nama_Cabang'])['Omset'].sum().reset_index()

fig11 = px.box(daily_branch, x='Nama_Cabang', y='Omset', labels={'Omset': 'Omset (Rp)', 'Nama_Cabang': ''})
fig11.update_traces(marker_color=COLORS['primary'])
styled(fig11, 450, 1000)
fig11.show()
save(fig11, '11_boxplot_cabang')

  Saved: 11_boxplot_cabang.png


In [14]:
print(f'\nSelesai! {len([f for f in os.listdir("charts") if f.endswith(".png")])} charts tersimpan di charts/')


Selesai! 12 charts tersimpan di charts/
